# Day 7 ML/RL Prep

5 basic mixed questions after the long weekend.

Instructions:
- Answer with reasoning, not just the final result.
- Keep explanations short and precise.
- If you use code, explain what each dimension means.


## Question 1: `squeeze` vs `unsqueeze`

You have `x` with shape `(B, T, 1) = (2, 4, 1)`.

What shape is `x.squeeze(-1)`?
What shape is `x.unsqueeze(-1)`?
Explain what each operation does to the dimensions.


In [9]:
# Question 1: Answer and reasoning
import torch 
import torch.nn.functional as F

x = torch.randn(2, 4, 1)
x.squeeze(-1).shape, x.unsqueeze(-1).shape

# squeeze(-1) will remove the trailing dimention. (2, 4, 1) -> (2, 4).
# unsqueeze(-1) will add a trailing dimention. (2, 4, 1) -> (2, 4, 1, 1)

(torch.Size([2, 4]), torch.Size([2, 4, 1, 1]))

## Question 2: `gather` with time dimension

You have:
- `logits`: shape `(B, T, A) = (2, 3, 4)`
- `actions`: shape `(B, T) = (2, 3)`

You want the chosen-action logit for each `(batch, time)` position.

If you use `gather` over the action dimension, what shape should the index tensor have?
What shape is the gathered result before and after squeezing?


In [25]:
# Question 2: Answer and reasoning
b, t, a = 2, 3, 4

logits = torch.randn(b, t, a)
actions = torch.randint(0, a, (b, t))
scores = torch.gather(logits, 2, actions.unsqueeze(-1)).squeeze(-1)
scores.shape

# The gather method requires both the input tensor and the index tensor to be the same shape. 
# Therefore, the actions tensor requires the addition of a trailing dimention so that it's shape matches the input. 
# Then you can gather over the actions dimention successfully, which will be at index 2 of the dimentions. B=0, T=1, A=2.
# In the code again I the squeeze the result back into the shape 2, 3 or (B,T).

torch.Size([2, 3])

## Question 3: `Categorical.log_prob` shapes

Consider:

```python
logits = torch.randn(2, 3, 4)   # (B, T, A)
actions = torch.randint(0, 4, (2, 3))  # (B, T)
dist = torch.distributions.Categorical(logits=logits)
logp = dist.log_prob(actions)
```

What shape is `logp`?
Why does the action dimension disappear?


In [31]:
# Question 3: Answer and reasoning

b, t, a = 2, 3, 4
logits = torch.randn(b, t, a)
actions = torch.randint(0, a, (b, t))
dist = torch.distributions.Categorical(logits=logits)
logp = dist.log_prob(actions)
logits.shape, actions.shape, logp.shape

# logp is of shape (2, 3) or (B, T).
# logp contains the log probabilites of taking the actions described in the actions tensor given the distribution. 
# therefore it is only of size (B, T) because there is one score representing probability at each timestep. 

(torch.Size([2, 3, 4]), torch.Size([2, 3]), torch.Size([2, 3]))

## Question 4: Debugging accidental squeeze

You have:
- `returns`: shape `(B, T, 1) = (2, 3, 1)`

Someone writes:

```python
returns = returns.squeeze()
```

What shape do they get here?
Why can using plain `squeeze()` be risky?
What would be safer if they only wanted to remove the last dimension?


In [32]:
# Question 4: Answer and reasoning
returns = torch.randn(2, 3, 1)
returns.shape, returns.squeeze().shape

# Squeeze with no index removes all dimentions with a size of 1. Which can lead to errors. 
# In the case at hand, as there is only one dimetion, squeeze and squeeze(-1) have the same effect. 
# But this is unsafe and instead you should always use squeeze(-1) even if there is only one size 1 dimention.

(torch.Size([2, 3, 1]), torch.Size([2, 3]))

## Question 5: Small masking implementation

You have:
- `x`: shape `(B, T, D) = (2, 3, 5)`
- `mask`: shape `(B, T) = (2, 3)` with 0/1 values

You want to compute the mean of valid feature vectors only, over batch and time.

Do not over-polish the code. I want:
- what shape the mask needs before multiplying `x`
- what the numerator should sum
- what the denominator should count
- what edge case you would handle


In [83]:
# Question 5: Answer and reasoning

b, t, d = 2, 3, 5
x = torch.randn(b, t, d)
mask = torch.randint(0, 2, (b, t))

mask_count = (mask != 0).sum().item()
masked_x = mask.unsqueeze(-1)*x

avg = masked_x.sum() / mask_count
avg

# To begin with x and the mask have different numbers of dimentions. 
# To allow for the multiplication to take place, a trailing dimention must be added to the mask, which will then be broadcasted
# accross the feature dimention. To do this we could use unsqueeze(-1).

# When calculating the average, the numerator should sum all the values in x that are not masked.
# In the code above I first masked x by multipling x*mask. This removes all the invalid features. 
# I then sum the valid features. 

# The denominator should count the number of 0's in the origonal mask.

# The code above does not handle the edge case, but it could add a very very small value to the mask_count to avoid division by 0.

tensor(1.4572)